# `aidp-fusion-bicc` live test — AIDP `aidataplatform` format (BICC PVO → Spark)
**Live-test row 10.** Uses AIDP's built-in `spark.read.format('aidataplatform')` connector — matches the official Oracle AIDP sample at `oracle-samples/oracle-aidp-samples` (`Read_Only_Ingestion_Connectors.ipynb`). The connector triggers the BICC extract, polls for completion, and reads the resulting CSVs from OCI Object Storage internally.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
from oracle_ai_data_platform_connectors.rest.fusion import read_bicc_via_aidp_format

# Prereqs: Fusion user has a BICC-admin role; an AIDP `EXTERNAL STORAGE`
# profile is registered in the catalog pointing at the OCI Object Storage
# bucket BICC writes to (configured once by an admin).
df = read_bicc_via_aidp_format(
    spark=spark,
    fusion_service_url=os.environ['FUSION_BICC_BASE_URL'],
    username=os.environ['FUSION_BICC_USER'],
    password=os.environ['FUSION_BICC_PASSWORD'],
    schema=os.environ['FUSION_BICC_SCHEMA'],            # e.g. 'ERP'
    datastore=os.environ['FUSION_BICC_PVO'],            # PVO name
    fusion_external_storage=os.environ['FUSION_BICC_EXTERNAL_STORAGE'],
)


In [ ]:
df.show(5)
print('rows:', df.count())
df.printSchema()


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-fusion-bicc',
    'auth': 'basic-aidp-format',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
